In [33]:
%%capture
!pip install -q promptlayer openai

In [34]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PROMPTLAYER_API_KEY"] = userdata.get("PROMPTLAYER_API_KEY")
os.environ["GROQ_KEY"] = userdata.get("GROQ_KEY")

In [35]:
%%capture
!pip install groq

In [36]:
from groq import Groq
import os

# client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
client = Groq(api_key = os.environ["GROQ_KEY"])

def sendMessage():
  response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me a coffee shop slogan."}
    ]
)
  return response.choices[0].message.content
responses = []
for i in range(3):
  responses.append(sendMessage())
for response in responses :
  print(response)

"Brewing up moments, one cup at a time."
"Brewing Joy, One Cup at a Time."
"Brewing up a better day, one cup at a time."


In [ ]:
print("Groq key prefix:", os.environ["GROQ_KEY"][:10])

Groq key prefix: gsk_HAOJX7


Trying to call PromptLayer but facing error

In [ ]:
from promptlayer import PromptLayer
from openai import OpenAI

pl_client = PromptLayer(api_key=os.environ["PROMPTLAYER_API_KEY"])

client = OpenAI(
    api_key = os.environ["GROQ_KEY"],
    base_url = "https://api.groq.com/openai/v1"
)


pl_response = pl_client.run(
    prompt_name = "coffee-shop-prompt",
    provider="Groq",  # exact provider name from PromptLayer
    model="llama-3.3-70b-versatile",
    input_variables= {}
)

request_id = pl_response["request_id"]
print("Request id ", request_id )

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Give me a coffee shop slogan."
        }
    ]
)

pl_client.track.score(
    request_id=1,
    score=100,
    score_name="success"
)


print(response.choices[0].message.content)

PromptLayerBadRequestError: PromptLayer had the following error while getting your prompt template: [{'type': 'literal_error', 'loc': ['provider'], 'msg': "Input should be 'anthropic', 'amazon.bedrock', 'cohere', 'google', 'huggingface', 'mistral', 'openai', 'openai.azure' or 'vertexai'"}]

#Extension Tasks

In [41]:

%%capture
!pip install langchain langchain-core langchain-groq langchain-openai rouge-score

In [42]:
from langchain_core.prompts import (
    PromptTemplate,
    FewShotPromptTemplate
)

from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq

from rouge_score import rouge_scorer

In [44]:
import os

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_KEY"],
    temperature=0.3
)

##Task 1: Earnings Call Summarization

Zero-Shot Prompt

In [39]:
import langchain
print(langchain.__version__)

1.3.4


In [45]:
earnings_generation_prompt = PromptTemplate(
    input_variables=["company", "quarter"],
    template="""
You are a financial analyst.

Generate a realistic earnings call transcript snippet for {company} during {quarter}.

Include:
- Revenue discussion
- Profitability discussion
- Future guidance
- Management commentary

Length: 150-200 words.
"""
)

generation_chain = (
    earnings_generation_prompt
    | llm
    | StrOutputParser()
)

snippet = generation_chain.invoke(
    {
        "company": "Microsoft",
        "quarter": "Q2 FY2025"
    }
)

print(snippet)

"Good morning, everyone. I'm Amy Hood, CFO of Microsoft. For Q2 FY2025, we reported revenue of $56.2 billion, up 10% year-over-year. Our cloud segment drove growth, with Azure revenue increasing 27%. 

Our operating income was $22.8 billion, representing a 12% increase. We saw strong profitability in our Productivity and Business Processes segment, with operating margins expanding to 43%. 

Looking ahead, we expect Q3 revenue to be between $54.5 and $56.5 billion. We're confident in our ability to drive long-term growth, with a focus on cloud, AI, and hybrid work solutions. 

As Satya mentioned, our investments in AI are paying off, with significant customer adoption of our Azure AI platform. We're well-positioned to capitalize on emerging trends and continue to deliver value to our shareholders."


Few-Shot Earnings Call Generation

In [46]:
examples = [
    {
        "company": "Apple",
        "quarter": "Q1 FY2025",
        "snippet": """
Revenue increased 8% year-over-year driven by strong iPhone and Services growth.
Operating margins improved due to cost optimization.
Management expects continued demand in the upcoming quarter.
"""
    },
    {
        "company": "Amazon",
        "quarter": "Q4 FY2024",
        "snippet": """
AWS continued double-digit growth while advertising revenue exceeded expectations.
Profitability improved through operational efficiencies.
Management provided positive guidance for the next quarter.
"""
    }
]
example_prompt = PromptTemplate.from_template(
"""
Company: {company}

Quarter: {quarter}

Transcript:
{snippet}
"""
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="""
You are a financial analyst.

Generate realistic earnings call snippets following the style of the examples.
""",
    suffix="""
Company: {company}

Quarter: {quarter}

Transcript:
""",
    input_variables=["company", "quarter"]
)
few_shot_chain = (
    few_shot_prompt
    | llm
    | StrOutputParser()
)
result = few_shot_chain.invoke(
    {
        "company": "NVIDIA",
        "quarter": "Q1 FY2026"
    }
)

print(result)

Company: NVIDIA

Quarter: Q1 FY2026

Transcript:

Revenue grew 12% year-over-year, driven by exceptional demand for our GeForce and datacenter products.
Gross margins expanded due to a favorable mix of high-end graphics cards and increasing adoption of our AI computing solutions.
Management expects sustained momentum in the fields of artificial intelligence, gaming, and cloud computing, and is confident in achieving strong growth throughout the fiscal year.


Zero-Shot Summarization

In [47]:
summary_prompt = PromptTemplate.from_template(
"""
Summarize the following earnings call transcript.

Include:

1. Revenue Performance
2. Profitability
3. Key Drivers
4. Future Outlook

Transcript:

{transcript}

Summary:
"""
)
summary_chain = (
    summary_prompt
    | llm
    | StrOutputParser()
)
summary = summary_chain.invoke(
    {
        "transcript": snippet
    }
)

print(summary)

Here is a summary of the earnings call transcript:

1. **Revenue Performance**: Microsoft reported revenue of $56.2 billion for Q2 FY2025, representing a 10% year-over-year increase, driven by strong growth in the cloud segment, particularly Azure, which saw a 27% revenue increase.

2. **Profitability**: The company's operating income was $22.8 billion, a 12% increase, with strong profitability in the Productivity and Business Processes segment, which had operating margins of 43%.

3. **Key Drivers**: The key drivers of Microsoft's growth were its cloud segment, particularly Azure, as well as its investments in AI, which are paying off with significant customer adoption of the Azure AI platform.

4. **Future Outlook**: Looking ahead, Microsoft expects Q3 revenue to be between $54.5 and $56.5 billion and is confident in its ability to drive long-term growth, with a focus on cloud, AI, and hybrid work solutions, positioning the company to capitalize on emerging trends and deliver value t

Few-Shot Summarization

In [48]:
summary_examples = [
    {
        "transcript": """
Revenue grew 15% due to cloud services.
Margins expanded because of cost efficiencies.
Management expects strong growth next quarter.
""",
        "summary": """
Revenue increased 15% driven by cloud growth.
Margins improved through operational efficiencies.
Management provided positive guidance.
"""
    }
]
summary_example_prompt = PromptTemplate.from_template(
"""
Transcript:
{transcript}

Summary:
{summary}
"""
)
few_shot_summary_prompt = FewShotPromptTemplate(
    examples=summary_examples,
    example_prompt=summary_example_prompt,
    prefix="""
Summarize earnings call transcripts.
""",
    suffix="""
Transcript:
{transcript}

Summary:
""",
    input_variables=["transcript"]
)
few_shot_summary_chain = (
    few_shot_summary_prompt
    | llm
    | StrOutputParser()
)
summary = few_shot_summary_chain.invoke(
    {
        "transcript": snippet
    }
)

print(summary)

Microsoft's Q2 FY2025 revenue was $56.2 billion, up 10% year-over-year, driven by cloud growth with Azure revenue increasing 27%. Operating income rose 12% to $22.8 billion, with expanding margins in the Productivity and Business Processes segment. The company expects Q3 revenue to be between $54.5 and $56.5 billion and is confident in its ability to drive long-term growth, fueled by investments in cloud, AI, and hybrid work solutions.


ROUGE-L Evaluation

In [49]:
reference_summary = """
Revenue increased due to cloud adoption.
Margins improved through cost efficiencies.
Management expects continued growth.
"""

generated_summary = summary
scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

scores = scorer.score(
    reference_summary,
    generated_summary
)

print(scores["rougeL"])

Score(precision=0.07894736842105263, recall=0.4, fmeasure=0.13186813186813187)


Generate 3 Earnings Call Snippets + Summaries + ROUGE-L Together

In [50]:
companies = [
    ("Microsoft", "Q2 FY2025"),
    ("NVIDIA", "Q1 FY2026"),
    ("Amazon", "Q4 FY2025")
]

for company, quarter in companies:

    transcript = generation_chain.invoke(
        {
            "company": company,
            "quarter": quarter
        }
    )

    summary = summary_chain.invoke(
        {
            "transcript": transcript
        }
    )

    print("=" * 80)
    print(company)
    print("=" * 80)

    print("\nTRANSCRIPT\n")
    print(transcript)

    print("\nSUMMARY\n")
    print(summary)

Microsoft

TRANSCRIPT

Amy Hood, CFO: "For Q2 FY2025, Microsoft reported revenue of $56.2 billion, up 10% year-over-year. Our cloud segment drove growth, with Azure revenue increasing 27%. 

Operating income was $22.8 billion, representing a 20% margin. We saw improved profitability in our Productivity and Business Processes segment, driven by Office 365 and LinkedIn.

Looking ahead, we expect Q3 revenue to be between $54.5 and $55.5 billion. We anticipate continued cloud growth, with Azure revenue expected to increase over 25% year-over-year.

Satya Nadella, CEO: "Our results demonstrate the strength of our cloud-first strategy. We're seeing significant traction with our Azure AI and machine learning offerings, and our Microsoft 365 suite continues to resonate with customers. We're well-positioned for long-term growth and remain committed to investing in innovation and customer success."

SUMMARY

Here's a summary of the earnings call transcript:

1. **Revenue Performance**: Microsoft

##TASK 2 - Ticket Classifier

Few-Shot Examples

In [51]:
ticket_examples = [
    {
        "ticket": "I was charged twice this month.",
        "category": "Billing"
    },
    {
        "ticket": "The application crashes when I login.",
        "category": "Technical"
    },
    {
        "ticket": "I cancelled my plan and need my money back.",
        "category": "Refund"
    },
    {
        "ticket": "Can you explain premium features?",
        "category": "General"
    },
    {
        "ticket": "Support has not resolved my issue after multiple attempts.",
        "category": "Escalate"
    }
]

In [52]:
ticket_example_prompt = PromptTemplate.from_template(
"""
Ticket:
{ticket}

Category:
{category}
"""
)

Few Shot Classification

In [53]:
ticket_prompt = FewShotPromptTemplate(
    examples=ticket_examples,
    example_prompt=ticket_example_prompt,
    prefix="""
You are a support ticket classifier.

Available categories:

- Billing
- Technical
- Refund
- General
- Escalate

Reason step-by-step internally and return only:

Category: <label>
""",
    suffix="""
Ticket:
{ticket}

Category:
""",
    input_variables=["ticket"]
)

Chain

In [54]:
ticket_chain = (
    ticket_prompt
    | llm
    | StrOutputParser()
)

Test

In [55]:
ticket = """
I have contacted support three times.
The issue remains unresolved and I need urgent attention.
"""

result = ticket_chain.invoke(
    {
        "ticket": ticket
    }
)

print(result)

Category: Escalate


#Provided by Rajat

In [56]:
%%capture
!pip install langchain langchain-community langchain-openai faiss-cpu pypdf

In [59]:
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_KEY")


from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def create_policy_qa_chain(pdf_path: str):

    # Load PDF
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    # Split into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)

    # Embeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # Vector Store
    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )

    # Groq LLM
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0
    )

    prompt = ChatPromptTemplate.from_template(
        """
You are an assistant for question-answering tasks.

Use only the provided context.

If the answer is not available in the context,
say "I don't know".

Context:
{context}

Question:
{question}
"""
    )

    def format_docs(docs):
        return "\n\n".join(
            doc.page_content for doc in docs
        )

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

In [64]:
pdf_file = "/content/Deep_TimeSeries_EDA_Report.pdf"

qa_chain = create_policy_qa_chain(pdf_file)

question = "What is the Descriptive Statistics in document?"

response = qa_chain.invoke(question)

print(response)

print()
print()

question = "What is the document regarding?"

response = qa_chain.invoke(question)

print(response)


print()
print()

question = "What is the total strength?"

response = qa_chain.invoke(question)

print(response)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The Descriptive Statistics are as follows:

- count: 7423.000000
- mean: 7.390210
- std: 88.544662
- min: -232.529984
- 25%: -54.934158
- 50%: -13.113405
- 75%: 73.771633
- max: 441.643674


The document appears to be a Deep Time Series Exploratory Data Analysis (EDA) Report, focusing on analyzing a time series dataset, specifically the "red_corrected" variable.


I don't know.


In [67]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))

In [69]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [70]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file == "PromptLayer.ipynb":
            print(os.path.join(root, file))

/content/drive/MyDrive/Colab Notebooks/PromptLayer.ipynb


In [71]:
import nbformat

notebook_path = "/content/drive/MyDrive/Colab Notebooks/PromptLayer.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open(notebook_path, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Widget metadata removed successfully.")

Widget metadata removed successfully.
